In [ ]:
!pip install nbstripout
!nbstripout "Copie de RAG_Finale.ipynb"


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!nbstripout "/content/drive/MyDrive/Copie de RAG_Finale.ipynb"


In [ ]:
# ================================================================
# CELLULE 1 - INSTALLATION ET GOOGLE DRIVE
# ================================================================

!pip install -q \
    langchain \
    langchain-community \
    sentence-transformers \
    transformers \
    accelerate \
    bitsandbytes \
    faiss-cpu \
    pypdf \
    torch \
    gradio



In [ ]:
!pip install --upgrade langchain langchain-community
!pip install --upgrade numpy torch transformers sentence-transformers faiss-cpu


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATA_PATH = "/content/drive/MyDrive/RAG/data "
import os
os.makedirs(DATA_PATH, exist_ok=True)

print("Drive monté et dépendances installées")


In [ ]:
pip install -U langchain-text-splitters


In [ ]:
# ================================================================
# CELLULE 2 - LLM, EMBEDDINGS ET DOCUMENTS (LANGCHAIN)
# ================================================================

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline


# ---------------------
# DEVICE
# ---------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device utilisé: {device}")

# ---------------------
# SOLUTION STABLE : Phi-2 (2.7B) - Excellent pour Colab
# ---------------------
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import gc

# Nettoyer d'abord la mémoire
gc.collect()
torch.cuda.empty_cache()

# Phi-2 de Microsoft - Très performant et stable
model_name = "microsoft/phi-2"

print(f" Chargement du modèle {model_name}...")


tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Chargement simple et stable
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    trust_remote_code=True,
    low_cpu_mem_usage=True
)

# Déplacer sur GPU si disponible
if torch.cuda.is_available():
    model = model.to("cuda")
    print(" Modèle chargé sur GPU")
else:
    print(" Modèle chargé sur CPU")

print(" Configuration du pipeline...")
generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.3,
    top_p=0.85,
    top_k=50,
    repetition_penalty=1.2,
    do_sample=True,
    return_full_text=False,
    pad_token_id=tokenizer.pad_token_id
)

llm = HuggingFacePipeline(pipeline=generation_pipeline)

print(f" Phi-2 chargé avec succès !")
print(f" Taille du modèle : 2.7B paramètres")
print(f" Mémoire GPU utilisée : {torch.cuda.memory_allocated() / 1e9:.2f} GB" if torch.cuda.is_available() else "")

# ---------------------
# EMBEDDINGS
# ---------------------
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
)

print("Embeddings chargés")

# ---------------------
# CHARGEMENT DES PDFS
# ---------------------
loader = PyPDFDirectoryLoader(DATA_PATH)
documents = loader.load()

print(f"{len(documents)} documents chargés")

# ---------------------
# SPLITTING
# ---------------------
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

splits = text_splitter.split_documents(documents)

print(f"{len(splits)} chunks créés")


In [ ]:
!ls "/content/drive/MyDrive/RAG/data "

In [ ]:
import langchain
import langchain_community
import numpy
import torch

print("LangChain version:", langchain.__version__)
print("LangChain-Community version:", langchain_community.__version__)
print("NumPy version:", numpy.__version__)
print("Torch version:", torch.__version__)


In [ ]:
# ================================================================
# CELLULE 3 - RAG LANGCHAIN + INTERFACE (CORRIGÉE - V3)
# ================================================================

import gradio as gr

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores.faiss import FAISS

# ---------------------
# VECTOR DATABASE (FAISS)
# ---------------------
vectorstore = FAISS.from_documents(
    splits,
    embeddings
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}
)

print("FAISS index créé")

# ---------------------
# RAG CHAIN (CORRIGÉE)
# ---------------------

# Définir le prompt EN FRANÇAIS
template = """
Tu es un assistant expert qui répond aux questions en te basant sur les documents fournis.
Utilise les informations du contexte ci-dessous pour répondre à la question.
Si tu ne trouves pas la réponse dans le contexte, dis-le clairement.
Réponds de manière concise et précise, en français uniquement.

Contexte : {context}

Question : {question}

Réponse détaillée en français :"""

prompt = ChatPromptTemplate.from_template(template)

# Fonction pour formater les documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Fonction wrapper pour appeler le LLM
def call_llm(prompt_value):
    """
    Appelle le LLM et retourne une chaîne de caractères propre
    """
    # Convertir le prompt en string
    if hasattr(prompt_value, 'to_string'):
        prompt_str = prompt_value.to_string()
    elif hasattr(prompt_value, 'text'):
        prompt_str = prompt_value.text
    else:
        prompt_str = str(prompt_value)

    # Appel au LLM
    response = llm.invoke(prompt_str)

    # Extraction de la réponse selon le format retourné
    if isinstance(response, str):
        return response
    elif isinstance(response, dict):
        for key in ['generated_text', 'text', 'content']:
            if key in response:
                return response[key]
        return str(list(response.values())[0]) if response else ""
    elif isinstance(response, list) and len(response) > 0:
        first = response[0]
        if isinstance(first, dict):
            for key in ['generated_text', 'text', 'content']:
                if key in first:
                    return first[key]
            return str(first)
        return str(first)
    elif hasattr(response, 'content'):
        return response.content
    else:
        return str(response)

# *** CORRECTION : Utiliser invoke au lieu de get_relevant_documents ***
def retrieve_docs(inputs):
    """
    Extrait la question et récupère les documents pertinents
    """
    # Si inputs est un dict avec une clé 'question'
    if isinstance(inputs, dict) and 'question' in inputs:
        query = inputs['question']
    else:
        query = str(inputs)

    # Utiliser invoke (nouvelle API) au lieu de get_relevant_documents
    docs = retriever.invoke(query)
    return docs

# Construire la chaîne RAG avec extraction explicite de la question
rag_chain = (
    {
        "context": RunnableLambda(retrieve_docs) | format_docs,
        "question": lambda x: x["question"] if isinstance(x, dict) else x
    }
    | prompt
    | call_llm
)

print("RAG chain prête")

# ---------------------
# CHAT FUNCTION
# ---------------------
def chat_interface(query):
    if not query.strip():
        return "Veuillez poser une question."

    try:
        # Appeler la chaîne avec la requête
        answer = rag_chain.invoke({"question": query})

        # Nettoyer la réponse (enlever le prompt si présent)
        if isinstance(answer, str):
            # Chercher différents marqueurs possibles
            for marker in ["Réponse détaillée en français :", "Réponse :", "Answer:"]:
                if marker in answer:
                    answer = answer.split(marker)[-1].strip()
                    break
            # Enlever la question si elle est répétée
            if query in answer:
                answer = answer.replace(query, "").strip()

        # Récupérer les documents sources séparément (avec invoke)
        docs = retriever.invoke(query)

        sources_text = "\n\n### 📚 Sources utilisées\n"
        seen_sources = set()

        for i, doc in enumerate(docs, 1):
            source = doc.metadata.get('source', 'Document inconnu')
            # Extraire juste le nom du fichier
            if '/' in source:
                source = source.split('/')[-1]

            # Éviter les doublons
            if source not in seen_sources:
                seen_sources.add(source)
                page = doc.metadata.get('page', 'N/A')
                sources_text += f"{len(seen_sources)}. **{source}** (page {page})\n"

        return f"### 💬 Réponse\n\n{answer}\n{sources_text}"

    except Exception as e:
        print(f"Erreur détaillée : {type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()
        return f"❌ **Une erreur est survenue**\n\n```\n{str(e)}\n```"

# ---------------------
# GRADIO INTERFACE
# ---------------------
with gr.Blocks(title="RAG LangChain avec Mistral-7B", theme=gr.themes.Soft()) as app:
    gr.Markdown("# 🤖 Assistant RAG Intelligent")
    gr.Markdown("### 💡 Posez vos questions sur vos documents PDF (Power BI, Big Data, JADE...)")

    with gr.Row():
        with gr.Column(scale=2):
            question = gr.Textbox(
                label="❓ Votre question",
                placeholder="Exemple : Quelle est la différence entre SUM et SUMX en DAX ?",
                lines=4
            )

            with gr.Row():
                submit = gr.Button("🚀 Générer la réponse", variant="primary", size="lg")
                clear = gr.Button("🗑️ Effacer", size="lg")

            gr.Markdown("**Exemples de questions :**")
            gr.Examples(
                examples=[
                    "Quelle est la différence entre SUM et SUMX en DAX ?",
                    "Qu'est-ce que Big Data Analytics ?",
                    "Explique-moi le framework JADE",
                    "Comment fonctionne Power BI ?",
                ],
                inputs=question
            )

    with gr.Row():
        output = gr.Markdown(label="📝 Réponse")

    submit.click(
        fn=chat_interface,
        inputs=question,
        outputs=output
    )

    clear.click(
        fn=lambda: ("", ""),
        inputs=None,
        outputs=[question, output]
    )

print("✅ Interface Gradio prête")
app.launch(debug=True, share=True)

In [ ]:
!pip install -q langchain-huggingface

In [ ]:
# Vérification des variables nécessaires
print("Variables présentes :")
print("- llm :", 'llm' in locals() or 'llm' in globals())
print("- retriever :", 'retriever' in locals() or 'retriever' in globals())
print("- rag_chain_with_source :", 'rag_chain_with_source' in locals() or 'rag_chain_with_source' in globals())
print("- vectorstore :", 'vectorstore' in locals() or 'vectorstore' in globals())

In [ ]:
!pip install -q ragas
